# 03 · API client

Talk to a running service (`uv run asl-api`, `docker compose -f docker/compose.yaml up api`,
or a Kubernetes ingress). `/v1/predict` classifies one image; `/v1/live` is a
WebSocket that takes a stream of encoded frames and answers each with the
per-frame and smoothed prediction plus hand landmarks.

In [ ]:
# Run from the repository root so params.yaml and data/ paths resolve.
import os
from pathlib import Path

if not Path("params.yaml").exists():
    os.chdir(Path.cwd().parent)
print(Path.cwd())

In [ ]:
import httpx

API = "http://localhost:8000"
print(httpx.get(f"{API}/ready").json())
info = httpx.get(f"{API}/v1/model").json()
info["name"], info["preprocess"]

In [ ]:
from pathlib import Path

image_path = next(Path("data/raw/Test").rglob("*.jpg"))
with open(image_path, "rb") as f:
    response = httpx.post(f"{API}/v1/predict", files={"image": (image_path.name, f, "image/jpeg")})
print(image_path)
response.json()

## Live stream

Frames from a webcam (or a video file: pass its path to `cv2.VideoCapture`) are
JPEG-encoded and sent one at a time; waiting for each answer keeps latency
bounded. The browser demo at `/` does the same from JavaScript.

In [ ]:
import json

import cv2
from websockets.sync.client import connect

capture = cv2.VideoCapture(0)
with connect(API.replace("http", "ws") + "/v1/live?smoothing=5") as ws:
    for _ in range(50):
        ok, frame = capture.read()
        if not ok:
            break
        ws.send(cv2.imencode(".jpg", frame)[1].tobytes())
        result = json.loads(ws.recv())
        smoothed = result["smoothed"]
        print(f"frame {result['frame']:3d}  {result['latency_ms']:6.1f} ms  "
              f"{smoothed['label'] or '-':>2} {smoothed['confidence']:.2f}")
capture.release()